# 02 - Phân tích EuroSAT và giao thức CNN

Notebook này chuẩn bị pipeline thí nghiệm cho EuroSAT và chỉ dùng Basic CNN để chọn siêu tham số ban đầu. Chưa dùng test set cho tuning; phần so sánh bốn kiến trúc ở cuối dùng giao thức đã đóng băng.


## 1. Định nghĩa bài toán

Bài toán là phân loại ảnh vệ tinh RGB của EuroSAT thành 10 lớp bằng các kiến trúc CNN được xây dựng từ đầu. Trong notebook này, mục tiêu không phải là so sánh mọi kiến trúc ngay lập tức, mà là chọn giao thức huấn luyện cấp dataset bằng validation evidence.


In [ ]:
from pathlib import Path
import json
import random
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from PIL import Image
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

SEED = 42
EXPECTED_PYTHON = "C:/Users/anhca/anaconda3/envs/tf312/python.exe"
actual_python = sys.executable.replace("\\", "/")
assert actual_python.lower() == EXPECTED_PYTHON.lower(), actual_python

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "AGENTS.md").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("A05 project root not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "datasets" / "eurosat" / "EuroSAT_RGB"
SPLIT_DIR = PROJECT_ROOT / "results" / "splits"
HP_DIR = PROJECT_ROOT / "results" / "hyperparameters"
FIG_DIR = PROJECT_ROOT / "results" / "figures" / "eurosat" / "hyperparameters"
EXAMPLE_DIR = PROJECT_ROOT / "results" / "figures" / "eurosat" / "examples"
for path in [SPLIT_DIR, HP_DIR, FIG_DIR, EXAMPLE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

print(
    {
        "python_executable": actual_python,
        "python_version": sys.version.split()[0],
        "tensorflow_version": tf.__version__,
        "keras_version": keras.__version__,
        "devices": [str(device) for device in tf.config.list_physical_devices()],
        "project_root": PROJECT_ROOT.name,
    }
)

{'python_executable': 'C:/Users/anhca/anaconda3/envs/tf312/python.exe', 'python_version': '3.12.14', 'tensorflow_version': '2.21.0', 'keras_version': '3.15.1', 'devices': ["PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')"], 'project_root': 'A05'}


## 2. Mô tả dataset

EuroSAT trong assignment này được đọc trực tiếp từ `datasets/eurosat/EuroSAT_RGB/`. Mỗi thư mục con là một lớp land-use/land-cover. Các ảnh là RGB `64x64`, nên input resolution là data-determined.


In [ ]:
classes = sorted([path.name for path in DATA_DIR.iterdir() if path.is_dir()])
class_to_index = {class_name: index for index, class_name in enumerate(classes)}
records = []
for class_name in classes:
    for image_path in sorted((DATA_DIR / class_name).glob("*.jpg")):
        records.append(
            {
                "relative_path": image_path.relative_to(PROJECT_ROOT).as_posix(),
                "class_name": class_name,
                "label": class_to_index[class_name],
            }
        )
manifest = pd.DataFrame(records)

shape_counts = {}
format_counts = {}
for relative_path in manifest["relative_path"]:
    with Image.open(PROJECT_ROOT / relative_path) as image:
        shape = (image.height, image.width, len(image.getbands()))
        shape_counts[shape] = shape_counts.get(shape, 0) + 1
        format_counts[image.format] = format_counts.get(image.format, 0) + 1

summary = {
    "sample_count": int(len(manifest)),
    "class_count": int(manifest["class_name"].nunique()),
    "shape_counts": {str(key): value for key, value in sorted(shape_counts.items())},
    "format_counts": format_counts,
}
print(json.dumps(summary, indent=2))
assert len(manifest) == 27000
assert manifest["class_name"].nunique() == 10
assert shape_counts == {(64, 64, 3): 27000}
manifest.head()

{
  "sample_count": 27000,
  "class_count": 10,
  "shape_counts": {
    "(64, 64, 3)": 27000
  },
  "format_counts": {
    "JPEG": 27000
  }
}


                                                 relative_path  class_name  label
0     datasets/eurosat/EuroSAT_RGB/AnnualCrop/AnnualCrop_1.jpg  AnnualCrop      0
1    datasets/eurosat/EuroSAT_RGB/AnnualCrop/AnnualCrop_10.jpg  AnnualCrop      0
2   datasets/eurosat/EuroSAT_RGB/AnnualCrop/AnnualCrop_100.jpg  AnnualCrop      0
3  datasets/eurosat/EuroSAT_RGB/AnnualCrop/AnnualCrop_1000.jpg  AnnualCrop      0
4  datasets/eurosat/EuroSAT_RGB/AnnualCrop/AnnualCrop_1001.jpg  AnnualCrop      0


## 3. Tóm tắt integrity của dataset

Kết quả audit khớp với dữ liệu thực tế: 27,000 ảnh, 10 lớp, tất cả ảnh đọc được bằng Pillow ở dạng RGB `64x64`. Vì kích thước `64x64` đến trực tiếp từ dữ liệu, không cần thí nghiệm chọn resolution cho EuroSAT.


## 4. Phân bố lớp

Phân bố lớp được kiểm tra trước khi chia dữ liệu. Vì các lớp không hoàn toàn bằng nhau, split phải stratified để giữ tỷ lệ lớp gần giống nhau trong train, validation, và test.


In [3]:
class_distribution = manifest.groupby("class_name").size().reset_index(name="count")
class_distribution

             class_name  count
0            AnnualCrop   3000
1                Forest   3000
2  HerbaceousVegetation   3000
3               Highway   2500
4            Industrial   2500
5               Pasture   2000
6         PermanentCrop   2500
7           Residential   3000
8                 River   2500
9               SeaLake   3000


## 5. Ảnh ví dụ

Một ảnh đại diện cho mỗi lớp được hiển thị để kiểm tra trực quan định dạng dữ liệu. Các ảnh chỉ được đọc; file gốc không bị sửa đổi.


In [4]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for axis, class_name in zip(axes.ravel(), classes):
    row = manifest[manifest["class_name"] == class_name].iloc[0]
    with Image.open(PROJECT_ROOT / row["relative_path"]) as image:
        axis.imshow(image.convert("RGB"))
    axis.set_title(class_name, fontsize=9)
    axis.axis("off")
fig.tight_layout()
example_path = EXAMPLE_DIR / "eurosat_examples.png"
fig.savefig(example_path, dpi=150)
plt.close(fig)
print({"example_figure": example_path.relative_to(PROJECT_ROOT).as_posix()})

{'example_figure': 'results/figures/eurosat/examples/eurosat_examples.png'}


## 6. Biểu diễn dữ liệu

Mỗi ảnh được biểu diễn thành tensor `float32` có shape `(64, 64, 3)`. Pixel được chuẩn hóa từ khoảng số nguyên `[0, 255]` về `[0, 1]` bằng phép chia 255. Nhãn lớp được mã hóa thành số nguyên để dùng với `SparseCategoricalCrossentropy`.


In [ ]:
def decode_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.ensure_shape(image, [64, 64, 3])
    image = tf.image.convert_image_dtype(image, tf.float32)
    return image, label


sample_tensor, sample_label = decode_image(
    str(PROJECT_ROOT / manifest.iloc[0]["relative_path"]),
    int(manifest.iloc[0]["label"]),
)
print(
    {
        "sample_shape": tuple(sample_tensor.shape),
        "dtype": sample_tensor.dtype.name,
        "min": float(tf.reduce_min(sample_tensor)),
        "max": float(tf.reduce_max(sample_tensor)),
        "label": int(sample_label),
    }
)

{'sample_shape': (64, 64, 3), 'dtype': 'float32', 'min': 0.3176470696926117, 'max': 0.8078432083129883, 'label': 0}


## 7. Chia train / validation / test

Split cố định dùng tỷ lệ 70% train, 15% validation, 15% test với seed 42. Đây là một quyết định thiết kế thí nghiệm: train đủ lớn để học, validation dùng để chọn siêu tham số, và test được giữ riêng cho đánh giá cuối.


In [ ]:
train_df, temp_df = train_test_split(
    manifest,
    test_size=0.30,
    random_state=SEED,
    stratify=manifest["label"],
)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"],
)

train_df = train_df.sort_values("relative_path").reset_index(drop=True)
val_df = val_df.sort_values("relative_path").reset_index(drop=True)
test_df = test_df.sort_values("relative_path").reset_index(drop=True)

train_df.to_csv(SPLIT_DIR / "eurosat_train.csv", index=False)
val_df.to_csv(SPLIT_DIR / "eurosat_val.csv", index=False)
test_df.to_csv(SPLIT_DIR / "eurosat_test.csv", index=False)

sets = [set(df["relative_path"]) for df in [train_df, val_df, test_df]]
overlap_count = len(sets[0] & sets[1]) + len(sets[0] & sets[2]) + len(sets[1] & sets[2])
accounted = sum(len(df) for df in [train_df, val_df, test_df])
split_distribution = pd.concat(
    [
        train_df.groupby("class_name").size().rename("train"),
        val_df.groupby("class_name").size().rename("validation"),
        test_df.groupby("class_name").size().rename("test"),
    ],
    axis=1,
).reset_index()

print(
    {
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df),
        "accounted": accounted,
        "overlap_count": overlap_count,
    }
)
assert accounted == 27000
assert overlap_count == 0
split_distribution

{'train': 18900, 'validation': 4050, 'test': 4050, 'accounted': 27000, 'overlap_count': 0}


             class_name  train  validation  test
0            AnnualCrop   2100         450   450
1                Forest   2100         450   450
2  HerbaceousVegetation   2100         450   450
3               Highway   1750         375   375
4            Industrial   1750         375   375
5               Pasture   1400         300   300
6         PermanentCrop   1750         375   375
7           Residential   2100         450   450
8                 River   1750         375   375
9               SeaLake   2100         450   450


## 8. Preprocessing

Pipeline dùng `tf.io.decode_image`, tức là giải mã dựa trên nội dung ảnh thay vì suy luận từ extension. Với EuroSAT hiện tại tất cả ảnh là JPEG, nhưng chiến lược này nhất quán với policy chung và an toàn cho các dataset ảnh khác.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def make_dataset(df, batch_size, shuffle=False, seed=SEED, augment=False):
    paths = [str(PROJECT_ROOT / path) for path in df["relative_path"].tolist()]
    labels = df["label"].astype("int32").to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(df), seed=seed, reshuffle_each_iteration=True
        )
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset


print({"pipeline": "tf.io.decode_image -> ensure_shape(64,64,3) -> float32 [0,1]"})

{'pipeline': 'tf.io.decode_image -> ensure_shape(64,64,3) -> float32 [0,1]'}


## 9. Lý do về data augmentation

EuroSAT là ảnh vệ tinh nhìn từ trên xuống. Với các lớp land-use như forest, river, residential hoặc annual crop, lật ngang, lật dọc và xoay vừa phải có thể vẫn giữ nhãn. Tuy vậy augmentation không được bật mặc định; nó phải được so sánh bằng validation evidence.


In [ ]:
from models.architectures import build_basic_cnn_2d

augmentation_layer = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal_and_vertical", seed=SEED),
        keras.layers.RandomRotation(0.10, fill_mode="reflect", seed=SEED),
    ],
    name="conservative_eurosat_augmentation",
)


def build_training_model(learning_rate, use_augmentation=False, dropout_rate=0.0):
    base = build_basic_cnn_2d(input_shape=(64, 64, 3), num_classes=len(classes))
    inputs = keras.Input(shape=(64, 64, 3), name="image")
    x = augmentation_layer(inputs) if use_augmentation else inputs
    outputs = base(x)
    model = keras.Model(inputs, outputs, name="basic_cnn_2d_training")
    if dropout_rate > 0:
        # Minimal regularization variant for an explicit overfitting check only.
        inner_inputs = keras.Input(shape=(64, 64, 3), name="image")
        y = keras.layers.Conv2D(32, 3, padding="same", activation="relu")(inner_inputs)
        y = keras.layers.MaxPooling2D(2)(y)
        y = keras.layers.Conv2D(64, 3, padding="same", activation="relu")(y)
        y = keras.layers.MaxPooling2D(2)(y)
        y = keras.layers.GlobalAveragePooling2D()(y)
        y = keras.layers.Dense(64, activation="relu")(y)
        y = keras.layers.Dropout(dropout_rate, seed=SEED)(y)
        outputs = keras.layers.Dense(len(classes), activation="softmax")(y)
        model = keras.Model(
            inner_inputs, outputs, name=f"basic_cnn_2d_dropout_{dropout_rate}"
        )
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


print(
    {
        "basic_cnn_parameters": int(
            build_basic_cnn_2d((64, 64, 3), len(classes)).count_params()
        )
    }
)

{'basic_cnn_parameters': 24202}


## 10. Khảo sát siêu tham số

Tất cả lựa chọn siêu tham số trong notebook này dùng Basic CNN 2D và validation set. Test set không xuất hiện trong tuning.

Chiến lược CPU-aware gồm hai giai đoạn:

- **Stage A - coarse screening**: dùng tuning subset stratified lấy từ train split, cùng subset cho mọi candidate.
- **Stage B - confirmation**: lấy 1-2 candidate tốt nhất từ Stage A và xác nhận bằng full train split với cùng validation split.

Tuning subset có 2,000 ảnh, tức 200 ảnh/lớp. Đây là **operational CPU constraint**, không phải kích thước tối ưu.


In [ ]:
def make_stratified_subset(df, per_class, seed=SEED):
    sampled_groups = []
    for class_name, group in df.groupby("class_name", sort=True):
        sampled_groups.append(
            group.sample(n=min(per_class, len(group)), random_state=seed)
        )
    return (
        pd.concat(sampled_groups, axis=0)
        .sort_values("relative_path")
        .reset_index(drop=True)
    )


tuning_train_df = make_stratified_subset(train_df, per_class=200, seed=SEED)
tuning_train_df.to_csv(SPLIT_DIR / "eurosat_tuning_train_subset.csv", index=False)
print(
    {
        "tuning_subset": len(tuning_train_df),
        "per_class_min": int(tuning_train_df.groupby("class_name").size().min()),
        "per_class_max": int(tuning_train_df.groupby("class_name").size().max()),
    }
)

{'tuning_subset': 2000, 'per_class_min': 200, 'per_class_max': 200}


In [ ]:
all_results = []


def evaluate_macro_f1(model, dataset):
    y_true_batches = []
    y_pred_batches = []
    for images, labels in dataset:
        probabilities = model.predict(images, verbose=0)
        y_true_batches.append(labels.numpy())
        y_pred_batches.append(np.argmax(probabilities, axis=1))
    y_true = np.concatenate(y_true_batches)
    y_pred = np.concatenate(y_pred_batches)
    return float(f1_score(y_true, y_pred, average="macro"))


def run_experiment(
    experiment_type,
    candidate_value,
    tuning_stage,
    train_source,
    train_frame,
    val_frame,
    learning_rate,
    batch_size,
    max_epochs,
    patience,
    use_augmentation=False,
    dropout_rate=0.0,
):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    train_ds = make_dataset(train_frame, batch_size=batch_size, shuffle=True, seed=SEED)
    val_ds = make_dataset(val_frame, batch_size=batch_size, shuffle=False)
    model = build_training_model(
        learning_rate=learning_rate,
        use_augmentation=use_augmentation,
        dropout_rate=dropout_rate,
    )
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=patience, restore_best_weights=True
        )
    ]
    start = time.perf_counter()
    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=max_epochs,
        callbacks=callbacks,
        verbose=0,
    )
    training_time = time.perf_counter() - start
    val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
    val_macro_f1 = evaluate_macro_f1(model, val_ds)
    train_loss = float(history.history["loss"][-1])
    train_accuracy = float(history.history["accuracy"][-1])
    epochs_run = len(history.history["loss"])
    best_epoch = int(np.argmin(history.history["val_loss"]) + 1)
    result = {
        "experiment_type": experiment_type,
        "candidate_value": candidate_value,
        "tuning_stage": tuning_stage,
        "train_source": train_source,
        "epochs_run": epochs_run,
        "best_epoch": best_epoch,
        "max_epochs": max_epochs,
        "patience": patience,
        "learning_rate": learning_rate,
        "batch_size": batch_size,
        "augmentation": use_augmentation,
        "dropout_rate": dropout_rate,
        "train_loss_last": train_loss,
        "train_accuracy_last": train_accuracy,
        "val_loss": float(val_loss),
        "val_accuracy": float(val_accuracy),
        "val_macro_f1": val_macro_f1,
        "training_time_seconds": float(training_time),
        "time_per_epoch_seconds": float(training_time / epochs_run),
        "seed": SEED,
    }
    all_results.append(result)
    pd.DataFrame(all_results).to_csv(
        HP_DIR / "eurosat_hyperparameters.csv", index=False
    )
    return result, history.history


print({"experiment_runner": "ready"})

{'experiment_runner': 'ready'}


### Stage A: sàng lọc learning rate

Learning rate được so sánh trên thang logarithmic cho Adam. Các candidate dùng cùng tuning subset, cùng validation split, cùng Basic CNN, và cùng ngân sách epoch ngắn.


In [ ]:
coarse_learning_rates = [1e-5, 3e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
coarse_histories = {}
for learning_rate in coarse_learning_rates:
    result, history = run_experiment(
        experiment_type="learning_rate",
        candidate_value=learning_rate,
        tuning_stage="A_coarse",
        train_source="stratified_train_subset_2000",
        train_frame=tuning_train_df,
        val_frame=val_df,
        learning_rate=learning_rate,
        batch_size=64,
        max_epochs=4,
        patience=2,
    )
    coarse_histories[str(learning_rate)] = history

coarse_lr_df = pd.DataFrame(
    [
        row
        for row in all_results
        if row["experiment_type"] == "learning_rate"
        and row["tuning_stage"] == "A_coarse"
    ]
)
coarse_lr_df.sort_values(["val_macro_f1", "val_loss"], ascending=[False, True])[
    [
        "candidate_value",
        "epochs_run",
        "best_epoch",
        "val_loss",
        "val_accuracy",
        "val_macro_f1",
        "training_time_seconds",
    ]
]

C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adap

   candidate_value  epochs_run  best_epoch  val_loss  val_accuracy  val_macro_f1  training_time_seconds
6          0.01000           4           4  1.697372      0.333086      0.271031              42.162767
4          0.00100           4           4  1.710441      0.339753      0.268852              65.117002
5          0.00300           4           4  1.703261      0.319012      0.240738              56.864016
3          0.00030           4           4  2.058332      0.269630      0.155023              70.912874
0          0.00001           4           4  2.300859      0.147161      0.049274               9.892034
2          0.00010           4           4  2.260463      0.092593      0.016949              69.324699
1          0.00003           4           4  2.294820      0.092593      0.016949              63.308421


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogx(coarse_lr_df["candidate_value"], coarse_lr_df["val_loss"], marker="o")
axes[0].set_xlabel("learning rate")
axes[0].set_ylabel("validation loss")
axes[0].set_title("Stage A LR vs validation loss")
axes[1].semilogx(
    coarse_lr_df["candidate_value"], coarse_lr_df["val_macro_f1"], marker="o"
)
axes[1].set_xlabel("learning rate")
axes[1].set_ylabel("validation macro F1")
axes[1].set_title("Stage A LR vs macro F1")
fig.tight_layout()
lr_plot_path = FIG_DIR / "learning_rate_stage_a.png"
fig.savefig(lr_plot_path, dpi=150)
plt.close(fig)

best_idx = int(
    coarse_lr_df.sort_values(
        ["val_macro_f1", "val_loss"], ascending=[False, True]
    ).index[0]
)
best_lr = float(coarse_lr_df.loc[best_idx, "candidate_value"])
boundary_note = "best candidate is internal"
if best_lr == min(coarse_learning_rates):
    boundary_note = "best candidate is lower boundary; lower rates would be considered if confirmation showed boundary pressure"
elif best_lr == max(coarse_learning_rates):
    boundary_note = (
        "best candidate is upper boundary; inspect instability before extending"
    )

print(
    {
        "stage_a_best_lr": best_lr,
        "boundary_note": boundary_note,
        "plot": lr_plot_path.relative_to(PROJECT_ROOT).as_posix(),
    }
)

{'stage_a_best_lr': 0.01, 'boundary_note': 'best candidate is upper boundary; inspect instability before extending', 'plot': 'results/figures/eurosat/hyperparameters/learning_rate_stage_a.png'}


### Stage B: xác nhận learning rate trên full train split

Hai learning-rate candidate mạnh nhất từ Stage A được xác nhận bằng full training split. Test set vẫn không được dùng.


In [ ]:
top_lr_candidates = (
    coarse_lr_df.sort_values(["val_macro_f1", "val_loss"], ascending=[False, True])[
        "candidate_value"
    ]
    .head(2)
    .astype(float)
    .tolist()
)
confirmation_histories = {}
for learning_rate in top_lr_candidates:
    result, history = run_experiment(
        experiment_type="learning_rate",
        candidate_value=learning_rate,
        tuning_stage="B_confirmation",
        train_source="full_train_split",
        train_frame=train_df,
        val_frame=val_df,
        learning_rate=learning_rate,
        batch_size=64,
        max_epochs=8,
        patience=3,
    )
    confirmation_histories[str(learning_rate)] = history

confirm_lr_df = pd.DataFrame(
    [
        row
        for row in all_results
        if row["experiment_type"] == "learning_rate"
        and row["tuning_stage"] == "B_confirmation"
    ]
)
selected_lr = float(
    confirm_lr_df.sort_values(
        ["val_macro_f1", "val_loss"], ascending=[False, True]
    ).iloc[0]["candidate_value"]
)
confirm_lr_df[
    [
        "candidate_value",
        "epochs_run",
        "best_epoch",
        "val_loss",
        "val_accuracy",
        "val_macro_f1",
        "training_time_seconds",
    ]
]

C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


   candidate_value  epochs_run  best_epoch  val_loss  val_accuracy  val_macro_f1  training_time_seconds
0            0.010           8           6  0.742875      0.721482      0.706063             556.890652
1            0.001           8           8  0.910257      0.671111      0.653650             637.379450


### Thí nghiệm batch size

Sau khi chọn learning rate bằng validation evidence, batch size được so sánh với các giá trị vừa phải phù hợp CPU/memory. Các biến khác được giữ cố định.


In [ ]:
batch_candidates = [16, 32, 64, 128]
batch_histories = {}
for batch_size in batch_candidates:
    result, history = run_experiment(
        experiment_type="batch_size",
        candidate_value=batch_size,
        tuning_stage="full_train_comparison",
        train_source="full_train_split",
        train_frame=train_df,
        val_frame=val_df,
        learning_rate=selected_lr,
        batch_size=batch_size,
        max_epochs=6,
        patience=2,
    )
    batch_histories[str(batch_size)] = history

batch_df = pd.DataFrame(
    [row for row in all_results if row["experiment_type"] == "batch_size"]
)
selected_batch = int(
    batch_df.sort_values(
        ["val_macro_f1", "val_loss", "time_per_epoch_seconds"],
        ascending=[False, True, True],
    ).iloc[0]["candidate_value"]
)
batch_df[
    [
        "candidate_value",
        "epochs_run",
        "best_epoch",
        "val_loss",
        "val_accuracy",
        "val_macro_f1",
        "time_per_epoch_seconds",
        "training_time_seconds",
    ]
]

C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adap

   candidate_value  epochs_run  best_epoch  val_loss  val_accuracy  val_macro_f1  time_per_epoch_seconds  training_time_seconds
0               16           6           6  0.744020      0.724198      0.709026               73.001576             438.009454
1               32           6           6  0.795092      0.710864      0.687747               55.513773             333.082641
2               64           6           6  0.742875      0.721482      0.706063               64.459978             386.759870
3              128           6           5  0.806096      0.710864      0.695698               68.694864             412.169185


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].plot(batch_df["candidate_value"], batch_df["val_macro_f1"], marker="o")
axes[0].set_xlabel("batch size")
axes[0].set_ylabel("validation macro F1")
axes[0].set_title("Batch size vs macro F1")
axes[1].plot(
    batch_df["candidate_value"], batch_df["time_per_epoch_seconds"], marker="o"
)
axes[1].set_xlabel("batch size")
axes[1].set_ylabel("seconds per epoch")
axes[1].set_title("Batch size vs epoch time")
fig.tight_layout()
batch_plot_path = FIG_DIR / "batch_size_comparison.png"
fig.savefig(batch_plot_path, dpi=150)
plt.close(fig)
print(
    {
        "selected_batch": selected_batch,
        "plot": batch_plot_path.relative_to(PROJECT_ROOT).as_posix(),
    }
)

{'selected_batch': 16, 'plot': 'results/figures/eurosat/hyperparameters/batch_size_comparison.png'}


### So sánh augmentation

Augmentation bảo thủ được so sánh với cấu hình Basic CNN đã chọn learning rate và batch size. Validation metrics quyết định có đưa augmentation vào selected protocol hay không.


In [ ]:
no_aug_match = (
    batch_df[batch_df["candidate_value"] == selected_batch]
    .sort_values(["val_macro_f1", "val_loss"], ascending=[False, True])
    .iloc[0]
    .to_dict()
)
aug_result, aug_history = run_experiment(
    experiment_type="augmentation",
    candidate_value="conservative_flip_rotation",
    tuning_stage="full_train_comparison",
    train_source="full_train_split",
    train_frame=train_df,
    val_frame=val_df,
    learning_rate=selected_lr,
    batch_size=selected_batch,
    max_epochs=6,
    patience=2,
    use_augmentation=True,
)
augmentation_df = pd.DataFrame([no_aug_match, aug_result])
augmentation_df["candidate_value"] = ["none", "conservative_flip_rotation"]
selected_augmentation = bool(
    aug_result["val_macro_f1"] > no_aug_match["val_macro_f1"]
    or (
        np.isclose(aug_result["val_macro_f1"], no_aug_match["val_macro_f1"])
        and aug_result["val_loss"] < no_aug_match["val_loss"]
    )
)
augmentation_df[
    [
        "candidate_value",
        "epochs_run",
        "best_epoch",
        "val_loss",
        "val_accuracy",
        "val_macro_f1",
        "training_time_seconds",
    ]
]

C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


              candidate_value  epochs_run  best_epoch  val_loss  val_accuracy  val_macro_f1  training_time_seconds
0                        none           6           6  0.744020      0.724198      0.709026             438.009454
1  conservative_flip_rotation           6           5  0.936012      0.661235      0.634755             552.497947


### Kiểm tra regularization

Dropout không được đặt tự động. Notebook kiểm tra khoảng cách train/validation từ run tốt nhất không augmentation. Chỉ khi có dấu hiệu overfitting rõ ràng mới cần thí nghiệm dropout.


In [ ]:
best_no_aug = no_aug_match
accuracy_gap = float(best_no_aug["train_accuracy_last"] - best_no_aug["val_accuracy"])
regularization_note = "No dropout experiment: overfitting evidence did not exceed the preset inspection threshold."
regularization_candidates_run = []
if accuracy_gap > 0.10:
    regularization_note = (
        "Accuracy gap exceeded 0.10; explicit dropout candidates were compared."
    )
    for dropout_rate in [0.2, 0.4]:
        result, history = run_experiment(
            experiment_type="dropout",
            candidate_value=dropout_rate,
            tuning_stage="full_train_regularization_check",
            train_source="full_train_split",
            train_frame=train_df,
            val_frame=val_df,
            learning_rate=selected_lr,
            batch_size=selected_batch,
            max_epochs=6,
            patience=2,
            dropout_rate=dropout_rate,
        )
        regularization_candidates_run.append(result)

print(
    {
        "train_val_accuracy_gap": accuracy_gap,
        "regularization_note": regularization_note,
        "dropout_candidates_run": len(regularization_candidates_run),
    }
)

{'train_val_accuracy_gap': -0.0010758042335510254, 'regularization_note': 'No dropout experiment: overfitting evidence did not exceed the preset inspection threshold.', 'dropout_candidates_run': 0}


## 11. Giao thức epoch

`max_epochs` là giới hạn vận hành, không phải tuyên bố số epoch tối ưu. EarlyStopping với `restore_best_weights=True` được dùng để dừng sớm khi validation loss không cải thiện.


In [ ]:
hp_path = HP_DIR / "eurosat_hyperparameters.csv"
results_df = pd.DataFrame(all_results)
results_df.to_csv(hp_path, index=False)

selected_augmentation_label = (
    "conservative_flip_rotation" if selected_augmentation else "none"
)
selected_run_pool = results_df[
    (results_df["learning_rate"] == selected_lr)
    & (results_df["batch_size"] == selected_batch)
]
selected_reference = selected_run_pool.sort_values(
    ["val_macro_f1", "val_loss"], ascending=[False, True]
).iloc[0]
selected_max_epochs = int(selected_reference["max_epochs"])
selected_patience = int(selected_reference["patience"])

protocol_rows = [
    {
        "decision": "input resolution",
        "selected_value": "64x64 RGB",
        "classification": "data-determined",
        "evidence_or_reason": "All EuroSAT files verified as 64x64 RGB.",
    },
    {
        "decision": "number of classes",
        "selected_value": len(classes),
        "classification": "data-determined",
        "evidence_or_reason": "10 class folders verified from actual files.",
    },
    {
        "decision": "train/validation/test split",
        "selected_value": "70/15/15 stratified, seed 42",
        "classification": "operational bound",
        "evidence_or_reason": "Provides training data, validation for selection, and isolated final test.",
    },
    {
        "decision": "model for hyperparameter selection",
        "selected_value": "Basic CNN 2D",
        "classification": "architecture-determined",
        "evidence_or_reason": "Assignment requires Basic CNN as the baseline for tuning before final architecture comparison.",
    },
    {
        "decision": "learning rate",
        "selected_value": selected_lr,
        "classification": "experimentally selected",
        "evidence_or_reason": "Selected from Stage B validation macro F1/loss.",
    },
    {
        "decision": "batch size",
        "selected_value": selected_batch,
        "classification": "experimentally selected",
        "evidence_or_reason": "Selected from validation metrics and time-per-epoch trade-off.",
    },
    {
        "decision": "augmentation",
        "selected_value": selected_augmentation_label,
        "classification": "experimentally selected",
        "evidence_or_reason": "Conservative augmentation kept only if validation evidence exceeded no augmentation.",
    },
    {
        "decision": "dropout",
        "selected_value": "not used unless dropout experiment selected it",
        "classification": "experimentally selected",
        "evidence_or_reason": regularization_note,
    },
    {
        "decision": "max_epochs",
        "selected_value": selected_max_epochs,
        "classification": "operational bound",
        "evidence_or_reason": "Upper bound paired with EarlyStopping; not claimed optimal.",
    },
    {
        "decision": "early stopping patience",
        "selected_value": selected_patience,
        "classification": "operational bound",
        "evidence_or_reason": "CPU-aware stopping rule based on validation loss stagnation.",
    },
]
protocol_df = pd.DataFrame(protocol_rows)
protocol_path = HP_DIR / "eurosat_selected_protocol.csv"
protocol_df.to_csv(protocol_path, index=False)
print(
    {
        "hyperparameter_results": hp_path.relative_to(PROJECT_ROOT).as_posix(),
        "selected_protocol": protocol_path.relative_to(PROJECT_ROOT).as_posix(),
        "experiment_rows": len(results_df),
    }
)
protocol_df

{'hyperparameter_results': 'results/hyperparameters/eurosat_hyperparameters.csv', 'selected_protocol': 'results/hyperparameters/eurosat_selected_protocol.csv', 'experiment_rows': 14}


                             decision                                  selected_value                           classification                                                                              evidence_or_reason
0                    input resolution                                       64x64 RGB                          data-determined                                                        All EuroSAT files verified as 64x64 RGB.
1                   number of classes                                              10                          data-determined                                                    10 class folders verified from actual files.
2         train/validation/test split                    70/15/15 stratified, seed 42         operational bound                      Provides training data, validation for selection, and isolated final test.
3  model for hyperparameter selection                                    Basic CNN 2D                  architecture-determi

## 12. Giao thức huấn luyện đã chọn

Bảng dưới đây là giao thức được chọn từ các thí nghiệm validation trong notebook này. Các mô hình cuối cùng của bốn kiến trúc chưa được huấn luyện ở phần này.


In [19]:
protocol_df

                             decision                                  selected_value                           classification                                                                              evidence_or_reason
0                    input resolution                                       64x64 RGB                          data-determined                                                        All EuroSAT files verified as 64x64 RGB.
1                   number of classes                                              10                          data-determined                                                    10 class folders verified from actual files.
2         train/validation/test split                    70/15/15 stratified, seed 42         operational bound                      Provides training data, validation for selection, and isolated final test.
3  model for hyperparameter selection                                    Basic CNN 2D                  architecture-determi

## 13. Tóm tắt

Notebook này đã xác minh dữ liệu EuroSAT từ file thật, tạo split stratified cố định, lưu membership CSV, xây dựng preprocessing không học từ test data, và chọn giao thức huấn luyện cấp dataset bằng Basic CNN trên validation set.


# 14. So sánh cuối cùng bốn mô hình

Các phần trước đã chọn giao thức huấn luyện cấp dataset cho EuroSAT chỉ bằng validation evidence. Phần cuối này đóng băng giao thức đó và chỉ thay đổi family kiến trúc: Basic CNN, AlexNet-inspired CNN, VGG-inspired CNN, và ResNet-inspired CNN. Test data chỉ được đánh giá sau khi cả bốn mô hình hoàn tất huấn luyện và chọn checkpoint.


In [ ]:
from pathlib import Path
import json
import random
import sys
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
import keras
from PIL import Image
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

from models.architectures import (
    build_alexnet_inspired_2d,
    build_basic_cnn_2d,
    build_resnet_inspired_2d,
    build_vgg_inspired_2d,
)

SEED = 42
EXPECTED_PYTHON = "C:/Users/anhca/anaconda3/envs/tf312/python.exe"
actual_python = sys.executable.replace("\\", "/")
assert actual_python.lower() == EXPECTED_PYTHON.lower(), actual_python

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "AGENTS.md").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("A05 project root not found")
    PROJECT_ROOT = PROJECT_ROOT.parent

SPLIT_DIR = PROJECT_ROOT / "results" / "splits"
HP_DIR = PROJECT_ROOT / "results" / "hyperparameters"
METRIC_DIR = PROJECT_ROOT / "results" / "metrics"
FIG_ROOT = PROJECT_ROOT / "results" / "figures" / "eurosat"
CURVE_DIR = FIG_ROOT / "training_curves"
CM_DIR = FIG_ROOT / "confusion_matrices"
ERROR_DIR = FIG_ROOT / "errors"
CHECKPOINT_DIR = PROJECT_ROOT / "models" / "checkpoints"
for path in [METRIC_DIR, CURVE_DIR, CM_DIR, ERROR_DIR, CHECKPOINT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(SPLIT_DIR / "eurosat_train.csv")
val_df = pd.read_csv(SPLIT_DIR / "eurosat_val.csv")
test_df = pd.read_csv(SPLIT_DIR / "eurosat_test.csv")
protocol_df = pd.read_csv(HP_DIR / "eurosat_selected_protocol.csv")
protocol = dict(zip(protocol_df["decision"], protocol_df["selected_value"]))

classes = sorted(train_df["class_name"].unique().tolist())
num_classes = len(classes)
label_to_class = dict(
    train_df[["label", "class_name"]].drop_duplicates().sort_values("label").values
)
class_names = [label_to_class[index] for index in range(num_classes)]

selected_lr = float(protocol["learning rate"])
selected_batch_size = int(protocol["batch size"])
selected_max_epochs = int(protocol["max_epochs"])
selected_patience = int(protocol["early stopping patience"])
selected_augmentation = str(protocol["augmentation"])
assert selected_augmentation == "none", selected_augmentation

sets = [set(df["relative_path"]) for df in [train_df, val_df, test_df]]
overlap_count = len(sets[0] & sets[1]) + len(sets[0] & sets[2]) + len(sets[1] & sets[2])
assert len(train_df) == 18900 and len(val_df) == 4050 and len(test_df) == 4050
assert overlap_count == 0

print(
    {
        "python_executable": actual_python,
        "tensorflow_version": tf.__version__,
        "devices": [str(device) for device in tf.config.list_physical_devices()],
        "train": len(train_df),
        "validation": len(val_df),
        "test": len(test_df),
        "learning_rate": selected_lr,
        "batch_size": selected_batch_size,
        "max_epochs": selected_max_epochs,
        "patience": selected_patience,
    }
)

{'python_executable': 'C:/Users/anhca/anaconda3/envs/tf312/python.exe', 'tensorflow_version': '2.21.0', 'devices': ["PhysicalDevice(name='/physical_device:CPU:0', device_type='CPU')"], 'train': 18900, 'validation': 4050, 'test': 4050, 'learning_rate': 0.01, 'batch_size': 16, 'max_epochs': 6, 'patience': 2}


## 15. Giao thức và data pipeline đã đóng băng

Phần so sánh dùng lại các file split train, validation, và test đã lưu. Preprocessing ảnh vẫn là `tf.io.decode_image`, convert RGB, kiểm tra shape cố định `64x64`, và scale pixel về `[0, 1]`. Augmentation policy được chọn là `none`, nên không kiến trúc nào nhận augmentation trong so sánh này.


In [ ]:
AUTOTUNE = tf.data.AUTOTUNE


def decode_image(path, label):
    image_bytes = tf.io.read_file(path)
    image = tf.io.decode_image(image_bytes, channels=3, expand_animations=False)
    image = tf.ensure_shape(image, [64, 64, 3])
    image = tf.image.convert_image_dtype(image, tf.float32)
    return image, label


def make_dataset(df, batch_size, shuffle=False, seed=SEED):
    paths = [str(PROJECT_ROOT / path) for path in df["relative_path"].tolist()]
    labels = df["label"].astype("int32").to_numpy()
    dataset = tf.data.Dataset.from_tensor_slices((paths, labels))
    if shuffle:
        dataset = dataset.shuffle(
            buffer_size=len(df), seed=seed, reshuffle_each_iteration=True
        )
    dataset = dataset.map(decode_image, num_parallel_calls=AUTOTUNE)
    dataset = dataset.batch(batch_size)
    dataset = dataset.prefetch(AUTOTUNE)
    return dataset


train_ds = make_dataset(train_df, selected_batch_size, shuffle=True, seed=SEED)
train_eval_ds = make_dataset(train_df, selected_batch_size, shuffle=False)
val_ds = make_dataset(val_df, selected_batch_size, shuffle=False)
test_ds = make_dataset(test_df, selected_batch_size, shuffle=False)

for sample_images, sample_labels in train_eval_ds.take(1):
    sample_shape = tuple(sample_images.shape[1:])
    sample_dtype = sample_images.dtype.name

print(
    {
        "sample_shape": sample_shape,
        "dtype": sample_dtype,
        "preprocessing": "tf.io.decode_image -> RGB -> float32 [0,1]",
    }
)

{'sample_shape': (64, 64, 3), 'dtype': 'float32', 'preprocessing': 'tf.io.decode_image -> RGB -> float32 [0,1]'}


## 16. Huấn luyện bốn family kiến trúc

Mỗi mô hình được huấn luyện từ đầu với Adam, learning rate đã chọn, batch size đã chọn, quy tắc EarlyStopping đã chọn, và seed 42. Validation loss điều khiển lựa chọn checkpoint. Không tuning riêng learning rate, batch size, dropout, hoặc epoch cho từng kiến trúc ở đây.


In [ ]:
MODEL_SPECS = [
    {
        "family": "BasicCNN2D",
        "key": "basic",
        "builder": build_basic_cnn_2d,
        "checkpoint": CHECKPOINT_DIR / "eurosat_basic.keras",
    },
    {
        "family": "AlexNetInspired2D",
        "key": "alexnet_inspired",
        "builder": build_alexnet_inspired_2d,
        "checkpoint": CHECKPOINT_DIR / "eurosat_alexnet_inspired.keras",
    },
    {
        "family": "VGGInspired2D",
        "key": "vgg_inspired",
        "builder": build_vgg_inspired_2d,
        "checkpoint": CHECKPOINT_DIR / "eurosat_vgg_inspired.keras",
    },
    {
        "family": "ResNetInspired2D",
        "key": "resnet_inspired",
        "builder": build_resnet_inspired_2d,
        "checkpoint": CHECKPOINT_DIR / "eurosat_resnet_inspired.keras",
    },
]


def compile_model(builder):
    model = builder(input_shape=(64, 64, 3), num_classes=num_classes)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=selected_lr),
        loss=keras.losses.SparseCategoricalCrossentropy(),
        metrics=[keras.metrics.SparseCategoricalAccuracy(name="accuracy")],
    )
    return model


def plot_history(history_df, family, key):
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].plot(history_df["epoch"], history_df["loss"], marker="o", label="train")
    axes[0].plot(
        history_df["epoch"], history_df["val_loss"], marker="o", label="validation"
    )
    axes[0].set_xlabel("epoch")
    axes[0].set_ylabel("loss")
    axes[0].set_title(f"{family}: loss")
    axes[0].legend()
    axes[1].plot(history_df["epoch"], history_df["accuracy"], marker="o", label="train")
    axes[1].plot(
        history_df["epoch"], history_df["val_accuracy"], marker="o", label="validation"
    )
    axes[1].set_xlabel("epoch")
    axes[1].set_ylabel("accuracy")
    axes[1].set_title(f"{family}: accuracy")
    axes[1].legend()
    fig.tight_layout()
    path = CURVE_DIR / f"{key}_history.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


training_rows = []
comparison_start = time.perf_counter()

for index, spec in enumerate(MODEL_SPECS):
    tf.keras.backend.clear_session()
    tf.keras.utils.set_random_seed(SEED)
    model = compile_model(spec["builder"])
    parameter_count = int(model.count_params())
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=selected_patience,
            restore_best_weights=True,
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=spec["checkpoint"],
            monitor="val_loss",
            save_best_only=True,
        ),
    ]
    train_model_ds = make_dataset(
        train_df, selected_batch_size, shuffle=True, seed=SEED
    )
    start = time.perf_counter()
    history = model.fit(
        train_model_ds,
        validation_data=val_ds,
        epochs=selected_max_epochs,
        callbacks=callbacks,
        verbose=0,
    )
    training_time = time.perf_counter() - start
    history_df = pd.DataFrame(history.history)
    history_df.insert(0, "epoch", np.arange(1, len(history_df) + 1))
    history_path = METRIC_DIR / f"eurosat_{spec['key']}_history.csv"
    history_df.to_csv(history_path, index=False)
    curve_path = plot_history(history_df, spec["family"], spec["key"])
    best_index = int(history_df["val_loss"].idxmin())
    best_epoch = int(history_df.loc[best_index, "epoch"])
    early_stopping_activated = len(history_df) < selected_max_epochs
    train_eval_loss, train_eval_accuracy = model.evaluate(train_eval_ds, verbose=0)
    val_loss, val_accuracy = model.evaluate(val_ds, verbose=0)
    training_rows.append(
        {
            "model_family": spec["family"],
            "key": spec["key"],
            "checkpoint_path": spec["checkpoint"].relative_to(PROJECT_ROOT).as_posix(),
            "parameter_count": parameter_count,
            "epochs_run": int(len(history_df)),
            "best_epoch": best_epoch,
            "early_stopping_activated": bool(early_stopping_activated),
            "train_loss_best_epoch_history": float(history_df.loc[best_index, "loss"]),
            "train_loss_final_epoch_history": float(history_df.iloc[-1]["loss"]),
            "train_accuracy_best_epoch_history": float(
                history_df.loc[best_index, "accuracy"]
            ),
            "train_accuracy_final_epoch_history": float(
                history_df.iloc[-1]["accuracy"]
            ),
            "train_eval_loss_best_weights": float(train_eval_loss),
            "train_eval_accuracy_best_weights": float(train_eval_accuracy),
            "validation_loss": float(val_loss),
            "validation_accuracy": float(val_accuracy),
            "training_time_seconds": float(training_time),
            "history_path": history_path.relative_to(PROJECT_ROOT).as_posix(),
            "history_figure": curve_path.relative_to(PROJECT_ROOT).as_posix(),
            "learning_rate": selected_lr,
            "batch_size": selected_batch_size,
            "max_epochs": selected_max_epochs,
            "patience": selected_patience,
            "seed": SEED,
        }
    )
    pd.DataFrame(training_rows).to_csv(
        METRIC_DIR / "eurosat_training_summaries.csv", index=False
    )
    print(
        {
            "trained": spec["family"],
            "parameters": parameter_count,
            "epochs_run": int(len(history_df)),
            "best_epoch": best_epoch,
            "early_stopping_activated": bool(early_stopping_activated),
            "training_time_seconds": round(training_time, 2),
            "completed": f"{index + 1}/{len(MODEL_SPECS)}",
        }
    )

training_summary_df = pd.DataFrame(training_rows)
training_summary_df[
    [
        "model_family",
        "parameter_count",
        "epochs_run",
        "best_epoch",
        "early_stopping_activated",
        "validation_loss",
        "validation_accuracy",
        "training_time_seconds",
    ]
]

{'trained': 'BasicCNN2D', 'parameters': 24202, 'epochs_run': 6, 'best_epoch': 6, 'early_stopping_activated': False, 'training_time_seconds': 365.39, 'completed': '1/4'}
{'trained': 'AlexNetInspired2D', 'parameters': 260170, 'epochs_run': 4, 'best_epoch': 2, 'early_stopping_activated': True, 'training_time_seconds': 677.58, 'completed': '2/4'}
{'trained': 'VGGInspired2D', 'parameters': 304810, 'epochs_run': 4, 'best_epoch': 2, 'early_stopping_activated': True, 'training_time_seconds': 1071.42, 'completed': '3/4'}
{'trained': 'ResNetInspired2D', 'parameters': 324490, 'epochs_run': 4, 'best_epoch': 2, 'early_stopping_activated': True, 'training_time_seconds': 1467.59, 'completed': '4/4'}


C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
C:\Users\anhca\anaconda3\envs\tf312\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adap

        model_family  ...  training_time_seconds
0         BasicCNN2D  ...             365.389023
1  AlexNetInspired2D  ...             677.584922
2      VGGInspired2D  ...            1071.420659
3   ResNetInspired2D  ...            1467.592041

[4 rows x 8 columns]

## 17. Đánh giá test sau huấn luyện

Chỉ sau khi cả bốn checkpoint đã được chọn bằng validation loss, phần này mới đánh giá test split được giữ riêng. Các metric thu được là số đo so sánh cuối cùng, không phải đầu vào cho quyết định siêu tham số sau đó.


In [ ]:
def predict_dataset(model, dataset):
    y_true_batches = []
    y_pred_batches = []
    probability_batches = []
    start = time.perf_counter()
    for images, labels in dataset:
        probabilities = model.predict(images, verbose=0)
        probability_batches.append(probabilities)
        y_true_batches.append(labels.numpy())
        y_pred_batches.append(np.argmax(probabilities, axis=1))
    inference_time = time.perf_counter() - start
    return (
        np.concatenate(y_true_batches),
        np.concatenate(y_pred_batches),
        np.concatenate(probability_batches),
        inference_time,
    )


def plot_confusion_matrix(matrix, family, key):
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(matrix, cmap="Blues")
    ax.set_xticks(np.arange(num_classes))
    ax.set_yticks(np.arange(num_classes))
    ax.set_xticklabels(class_names, rotation=45, ha="right", fontsize=8)
    ax.set_yticklabels(class_names, fontsize=8)
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_title(f"{family}: test confusion matrix")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    for row in range(num_classes):
        for col in range(num_classes):
            value = int(matrix[row, col])
            if value:
                ax.text(col, row, value, ha="center", va="center", fontsize=7)
    fig.tight_layout()
    path = CM_DIR / f"{key}_confusion_matrix.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


def top_confusion_pairs(matrix, top_n=5):
    rows = []
    for true_label in range(num_classes):
        for predicted_label in range(num_classes):
            if true_label == predicted_label:
                continue
            count = int(matrix[true_label, predicted_label])
            if count:
                rows.append(
                    {
                        "true_label": true_label,
                        "predicted_label": predicted_label,
                        "true_class": class_names[true_label],
                        "predicted_class": class_names[predicted_label],
                        "count": count,
                    }
                )
    return sorted(
        rows, key=lambda row: (-row["count"], row["true_class"], row["predicted_class"])
    )[:top_n]


def save_error_grid(prediction_df, pairs, family, key, examples_per_pair=4):
    selected = []
    for pair in pairs[:3]:
        pair_rows = (
            prediction_df[
                (prediction_df["true_label"] == pair["true_label"])
                & (prediction_df["predicted_label"] == pair["predicted_label"])
            ]
            .sort_values("relative_path")
            .head(examples_per_pair)
        )
        selected.extend(pair_rows.to_dict(orient="records"))
    if not selected:
        return None
    columns = examples_per_pair
    rows = int(np.ceil(len(selected) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(columns * 2.2, rows * 2.4))
    axes = np.array(axes).reshape(-1)
    for axis in axes:
        axis.axis("off")
    for axis, row in zip(axes, selected):
        with Image.open(PROJECT_ROOT / row["relative_path"]) as image:
            axis.imshow(image.convert("RGB"))
        axis.set_title(
            f"T: {row['true_class']}\nP: {row['predicted_class']}", fontsize=7
        )
        axis.axis("off")
    fig.suptitle(
        f"{family}: deterministic samples from top confusion pairs", fontsize=11
    )
    fig.tight_layout()
    path = ERROR_DIR / f"{key}_top_confusion_errors.png"
    fig.savefig(path, dpi=150)
    plt.close(fig)
    return path


metric_rows = []
per_class_rows = []
confusion_pair_rows = []
prediction_frames = {}

for spec in MODEL_SPECS:
    tf.keras.backend.clear_session()
    model = keras.models.load_model(spec["checkpoint"])
    test_loss, test_accuracy = model.evaluate(test_ds, verbose=0)
    y_true, y_pred, probabilities, inference_time = predict_dataset(model, test_ds)
    macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(num_classes)),
        average="macro",
        zero_division=0,
    )
    per_precision, per_recall, per_f1, per_support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=list(range(num_classes)),
        average=None,
        zero_division=0,
    )
    matrix = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))
    cm_path = plot_confusion_matrix(matrix, spec["family"], spec["key"])
    pairs = top_confusion_pairs(matrix, top_n=8)
    prediction_df = test_df.copy().reset_index(drop=True)
    prediction_df["true_label"] = y_true.astype(int)
    prediction_df["predicted_label"] = y_pred.astype(int)
    prediction_df["true_class"] = prediction_df["true_label"].map(label_to_class)
    prediction_df["predicted_class"] = prediction_df["predicted_label"].map(
        label_to_class
    )
    prediction_df["max_probability"] = probabilities.max(axis=1)
    prediction_path = METRIC_DIR / f"eurosat_{spec['key']}_test_predictions.csv"
    prediction_df.to_csv(prediction_path, index=False)
    prediction_frames[spec["key"]] = prediction_df
    error_grid_path = save_error_grid(
        prediction_df[prediction_df["true_label"] != prediction_df["predicted_label"]],
        pairs,
        spec["family"],
        spec["key"],
    )
    training_info = (
        training_summary_df[training_summary_df["key"] == spec["key"]].iloc[0].to_dict()
    )
    metric_rows.append(
        {
            "model_family": spec["family"],
            "key": spec["key"],
            "trainable_parameter_count": int(training_info["parameter_count"]),
            "epochs_run": int(training_info["epochs_run"]),
            "best_epoch": int(training_info["best_epoch"]),
            "early_stopping_activated": bool(training_info["early_stopping_activated"]),
            "train_loss_best_epoch_history": float(
                training_info["train_loss_best_epoch_history"]
            ),
            "train_loss_final_epoch_history": float(
                training_info["train_loss_final_epoch_history"]
            ),
            "train_eval_loss_best_weights": float(
                training_info["train_eval_loss_best_weights"]
            ),
            "validation_loss": float(training_info["validation_loss"]),
            "test_loss": float(test_loss),
            "test_accuracy": float(test_accuracy),
            "macro_precision": float(macro_precision),
            "macro_recall": float(macro_recall),
            "macro_f1": float(macro_f1),
            "training_time_seconds": float(training_info["training_time_seconds"]),
            "inference_time_seconds": float(inference_time),
            "inference_ms_per_sample": float(inference_time / len(test_df) * 1000),
            "checkpoint_path": training_info["checkpoint_path"],
            "confusion_matrix_figure": cm_path.relative_to(PROJECT_ROOT).as_posix(),
            "error_grid_figure": (
                ""
                if error_grid_path is None
                else error_grid_path.relative_to(PROJECT_ROOT).as_posix()
            ),
            "prediction_path": prediction_path.relative_to(PROJECT_ROOT).as_posix(),
        }
    )
    for class_index, class_name in enumerate(class_names):
        per_class_rows.append(
            {
                "model_family": spec["family"],
                "key": spec["key"],
                "class_label": class_index,
                "class_name": class_name,
                "precision": float(per_precision[class_index]),
                "recall": float(per_recall[class_index]),
                "f1": float(per_f1[class_index]),
                "support": int(per_support[class_index]),
            }
        )
    for rank, pair in enumerate(pairs, start=1):
        confusion_pair_rows.append(
            {"model_family": spec["family"], "key": spec["key"], "rank": rank, **pair}
        )
    print(
        {
            "evaluated": spec["family"],
            "test_accuracy": round(float(test_accuracy), 4),
            "macro_f1": round(float(macro_f1), 4),
            "inference_time_seconds": round(float(inference_time), 2),
        }
    )

models_df = pd.DataFrame(metric_rows)
per_class_df = pd.DataFrame(per_class_rows)
confusion_pairs_df = pd.DataFrame(confusion_pair_rows)

models_path = METRIC_DIR / "eurosat_models.csv"
per_class_path = METRIC_DIR / "eurosat_per_class_metrics.csv"
pairs_path = METRIC_DIR / "eurosat_confusion_pairs.csv"
models_df.to_csv(models_path, index=False)
per_class_df.to_csv(per_class_path, index=False)
confusion_pairs_df.to_csv(pairs_path, index=False)

total_comparison_runtime = time.perf_counter() - comparison_start
runtime_path = METRIC_DIR / "eurosat_final_runtime.json"
runtime_path.write_text(
    json.dumps(
        {
            "total_runtime_seconds": total_comparison_runtime,
            "model_training_seconds": models_df[
                ["model_family", "training_time_seconds"]
            ].to_dict(orient="records"),
            "model_inference_seconds": models_df[
                ["model_family", "inference_time_seconds"]
            ].to_dict(orient="records"),
            "python_executable": actual_python,
            "tensorflow_version": tf.__version__,
        },
        indent=2,
    ),
    encoding="utf-8",
)

models_df[
    [
        "model_family",
        "best_epoch",
        "validation_loss",
        "test_loss",
        "test_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "trainable_parameter_count",
        "training_time_seconds",
        "inference_time_seconds",
        "early_stopping_activated",
    ]
]

{'evaluated': 'BasicCNN2D', 'test_accuracy': 0.7398, 'macro_f1': 0.7256, 'inference_time_seconds': 40.82}
{'evaluated': 'AlexNetInspired2D', 'test_accuracy': 0.1111, 'macro_f1': 0.02, 'inference_time_seconds': 82.14}
{'evaluated': 'VGGInspired2D', 'test_accuracy': 0.1111, 'macro_f1': 0.02, 'inference_time_seconds': 64.27}
{'evaluated': 'ResNetInspired2D', 'test_accuracy': 0.1111, 'macro_f1': 0.02, 'inference_time_seconds': 16.27}


        model_family  ...  early_stopping_activated
0         BasicCNN2D  ...                     False
1  AlexNetInspired2D  ...                      True
2      VGGInspired2D  ...                      True
3   ResNetInspired2D  ...                      True

[4 rows x 12 columns]

## 18. Trade-off về độ phức tạp

Bảng và hình bên dưới so sánh hiệu năng đo được với số tham số, thời gian huấn luyện, và thời gian inference. Mô hình có test score cao nhất không tự động là lựa chọn tốt nhất; cần xét trade-off chi phí/hiệu năng đo được.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(models_df["trainable_parameter_count"], models_df["macro_f1"], s=80)
for _, row in models_df.iterrows():
    axes[0].annotate(
        row["model_family"].replace("Inspired2D", ""),
        (row["trainable_parameter_count"], row["macro_f1"]),
        fontsize=8,
    )
axes[0].set_xscale("log")
axes[0].set_xlabel("trainable parameters (log scale)")
axes[0].set_ylabel("test macro F1")
axes[0].set_title("Performance vs parameter count")

axes[1].scatter(models_df["training_time_seconds"], models_df["macro_f1"], s=80)
for _, row in models_df.iterrows():
    axes[1].annotate(
        row["model_family"].replace("Inspired2D", ""),
        (row["training_time_seconds"], row["macro_f1"]),
        fontsize=8,
    )
axes[1].set_xlabel("training time (seconds)")
axes[1].set_ylabel("test macro F1")
axes[1].set_title("Performance vs training time")
fig.tight_layout()
complexity_path = FIG_ROOT / "eurosat_complexity_tradeoffs.png"
fig.savefig(complexity_path, dpi=150)
plt.close(fig)

display_table = models_df[
    [
        "model_family",
        "test_accuracy",
        "macro_f1",
        "trainable_parameter_count",
        "training_time_seconds",
        "inference_ms_per_sample",
        "best_epoch",
        "early_stopping_activated",
    ]
].copy()
display_table["test_accuracy"] = display_table["test_accuracy"].round(4)
display_table["macro_f1"] = display_table["macro_f1"].round(4)
display_table["training_time_seconds"] = display_table["training_time_seconds"].round(2)
display_table["inference_ms_per_sample"] = display_table[
    "inference_ms_per_sample"
].round(4)

print(
    {
        "complexity_figure": complexity_path.relative_to(PROJECT_ROOT).as_posix(),
        "total_runtime_seconds": round(total_comparison_runtime, 2),
    }
)
display_table

{'complexity_figure': 'results/figures/eurosat/eurosat_complexity_tradeoffs.png', 'total_runtime_seconds': 4240.47}


        model_family  test_accuracy  ...  best_epoch  early_stopping_activated
0         BasicCNN2D         0.7398  ...           6                     False
1  AlexNetInspired2D         0.1111  ...           2                      True
2      VGGInspired2D         0.1111  ...           2                      True
3   ResNetInspired2D         0.1111  ...           2                      True

[4 rows x 8 columns]

## 19. Phân tích lỗi

Bảng confusion-pair được tính từ từng test confusion matrix. Các error grid là mẫu deterministic từ những cặp nhầm lẫn off-diagonal thường gặp nhất, sắp theo file path thay vì chọn thủ công.


In [ ]:
top_pairs_table = (
    confusion_pairs_df.groupby("model_family").head(5).reset_index(drop=True)
)
top_pairs_table[["model_family", "rank", "true_class", "predicted_class", "count"]]

         model_family  rank            true_class       predicted_class  count
0          BasicCNN2D     1            Industrial           Residential    104
1          BasicCNN2D     2               Highway                 River     93
2          BasicCNN2D     3                 River               Highway     82
3          BasicCNN2D     4         PermanentCrop               Highway     76
4          BasicCNN2D     5         PermanentCrop  HerbaceousVegetation     72
5   AlexNetInspired2D     1                Forest            AnnualCrop    450
6   AlexNetInspired2D     2  HerbaceousVegetation            AnnualCrop    450
7   AlexNetInspired2D     3           Residential            AnnualCrop    450
8   AlexNetInspired2D     4               SeaLake            AnnualCrop    450
9   AlexNetInspired2D     5               Highway            AnnualCrop    375
10      VGGInspired2D     1                Forest            AnnualCrop    450
11      VGGInspired2D     2  HerbaceousVegetation   

## 20. Ghi chú so sánh đo được

Với giao thức EuroSAT đã đóng băng, BasicCNN2D là mô hình duy nhất học được classifier hữu ích: test accuracy `0.7398`, macro F1 `0.7256`, 24,202 trainable parameters, và 365.39 giây huấn luyện. AlexNetInspired2D, VGGInspired2D, và ResNetInspired2D đều EarlyStopping sau 4 epoch với best epoch 2 và hội tụ về hành vi test suy biến giống nhau: mọi ảnh test được dự đoán là AnnualCrop, tạo test accuracy `0.1111` và macro F1 `0.0200`. Vì so sánh kiến trúc đóng băng giao thức cấp dataset, các kết quả này được ghi nhận như hành vi quan sát được, không dùng để retune riêng từng kiến trúc.

Confusion matrix của BasicCNN2D cho thấy các off-diagonal count lớn nhất là Industrial -> Residential, Highway -> River, và River -> Highway. Error grid deterministic cho thấy mẫu Industrial/Residential đều có cấu trúc xây dựng dày đặc, còn lỗi Highway/River thường có đặc trưng tuyến tính dài hoặc đường nước/đường bộ cắt qua vùng đất xung quanh tương tự. Với ba mô hình sâu hơn, pattern nhầm lẫn chính không phải là một cặp lớp tinh tế: confusion matrix cho thấy mô hình collapse về dự đoán AnnualCrop trên các lớp khác.

Trade-off độ phức tạp vì vậy không phải là mô hình lớn nhất tốt nhất. BasicCNN2D vừa là mô hình nhỏ nhất vừa là mô hình duy nhất thành công dưới giao thức đóng băng này. Các mô hình lớn hơn cần nhiều thời gian huấn luyện và nhiều tham số hơn, nhưng protocol learning rate đã chọn không tạo hành vi validation hoặc test hữu ích cho chúng. Đây là kết quả bất ngờ cần giữ lại trong thảo luận assignment; không được sửa bằng cách dùng test set hoặc tuning riêng theo kiến trúc trong notebook này.
